In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import igraph as ig
from networkx.algorithms import bipartite
import seaborn as sns

# Hyperlink

In [ ]:
# Hyperlink
# Load the data
h_nodes = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='nodes')
h_edges = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='edges')
# remove rows where Source == Target
h_edges = h_edges[h_edges.Source != h_edges.Target]
h_edges.reset_index(drop=True, inplace=True)
seeds_nodes = h_nodes[h_nodes['STATUS']=='IN']
#seeds_nodes['Stance']=[0,0,0,1,1,0,1,0,1,1,1,0,1,1,1,1]

In [ ]:
seeds_nodes = seeds_nodes[['ID','Label', 'Organization Name', 'Category']] #, 'Stance']]

In [ ]:
# assign a differnet color for each value of h_nodes['Category'].unique()
color_map_cat = {'GOV': 'red', 'ROE': 'yellow', 'PEC': 'green', 'ASN': 'orange', 'NGO': 'blue', 'OTHER': 'black'}

In [ ]:
# Create a directed graph for the Hyperlink data
G_h = nx.DiGraph()

for index, row in h_nodes.iterrows():
    G_h.add_node(row['ID'], label=row['Label'], status=row['STATUS'], category=row['Category'])

G_h.add_edges_from(h_edges[['Source', 'Target']].values)

In [ ]:
# create a directed graph among only nodes with 'STATUS' == 'IN'
h_net_seeds = nx.DiGraph()
h_net_sd = nx.DiGraph()
for i in range(h_nodes.shape[0]):
    h_net_sd.add_node(h_nodes['ID'][i])
    if h_nodes['STATUS'][i] == 'IN':
        h_net_seeds.add_node(h_nodes['ID'][i])
for i in range(h_edges.shape[0]):
    if h_edges['Source'][i] in h_net_seeds.nodes() and h_edges['Target'][i] in h_net_seeds.nodes():
        h_net_seeds.add_edge(h_edges['Source'][i], h_edges['Target'][i])
    else:
        h_net_sd.add_edge(h_edges['Source'][i], h_edges['Target'][i])

# remove self loops
h_net_seeds.remove_edges_from(nx.selfloop_edges(h_net_seeds))

In [ ]:
# compute the degree of each node in h_net_sd
h_net_sd_indegree = dict(h_net_sd.in_degree())

# compute the out-degree of each node in h_net_sd
h_net_sd_outdegree = dict(h_net_sd.out_degree())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

# --- Select only seed nodes (STATUS == "IN") from G_h ---
#seed_nodes = [node for node, data in G_h.nodes(data=True) if data.get('STATUS') == 'IN']

# --- Compute in- and out-degrees for seeds in G_h ---
h_seeds_indegree = {node: G_h.in_degree(node) for node in seeds_nodes['ID']}
h_seeds_outdegree = {node: G_h.out_degree(node) for node in seeds_nodes['ID']}

# ---------- In-degree Plot for Seeds ----------

# Sort the indegree dictionary in descending order
sorted_indegree = sorted(h_seeds_indegree.items(), key=lambda x: x[1], reverse=True)
sorted_ids_indegree = [node for node, deg in sorted_indegree]
sorted_values_indegree = [deg for node, deg in sorted_indegree]

# Use node IDs as x labels
x_labels_indegree = sorted_ids_indegree

# Get colors based on each node's category (lookup in h_nodes DataFrame)
colors_indegree = [
    color_map_cat[h_nodes[h_nodes['ID'] == node]['Category'].values[0]] 
    for node in sorted_ids_indegree
]



# ---------- Out-degree Plot for Seeds ----------

# Sort the outdegree dictionary in descending order
sorted_outdegree = sorted(h_seeds_outdegree.items(), key=lambda x: x[1], reverse=True)
sorted_ids_outdegree = [node for node, deg in sorted_outdegree]
sorted_values_outdegree = [deg for node, deg in sorted_outdegree]

# Use node IDs as x labels
x_labels_outdegree = sorted_ids_outdegree

# Get colors based on each node's category
colors_outdegree = [
    color_map_cat[h_nodes[h_nodes['ID'] == node]['Category'].values[0]] 
    for node in sorted_ids_outdegree
]


In [ ]:
# Ensure you are working on a copy of the DataFrame to avoid the warning
seeds_nodes = seeds_nodes.copy()

# Use .loc to explicitly assign values
seeds_nodes.loc[:, 'In-degree Tot'] = seeds_nodes['ID'].map(h_seeds_indegree)
seeds_nodes.loc[:, 'Out-degree Tot'] = seeds_nodes['ID'].map(h_seeds_outdegree)



In [ ]:
seeds_nodes

In [ ]:
# compute the degree of each node in G_h
h_indegree = dict(G_h.in_degree())
h_outdegree = dict(G_h.out_degree())

# compute the degree of each node in h_net_seeds
h_net_seeds_indegree = dict(h_net_seeds.in_degree())
h_net_seeds_outdegree = dict(h_net_seeds.out_degree())

# compute the degree of each node in h_net_sd
h_net_sd_indegree = dict(h_net_sd.in_degree())
h_net_sd_outdegree = dict(h_net_sd.out_degree())


# add the in-degree and out-degree of each node in G_h to the seeds_nodes DataFrame
seeds_nodes['In-degree'] = seeds_nodes['ID'].map(h_indegree)
seeds_nodes['Out-degree'] = seeds_nodes['ID'].map(h_outdegree)

# add the in-degree and out-degree of each node in h_net_seeds to the seeds_nodes DataFrame
seeds_nodes['In-degree Seeds'] = seeds_nodes['ID'].map(h_net_seeds_indegree)
seeds_nodes['Out-degree Seeds'] = seeds_nodes['ID'].map(h_net_seeds_outdegree)

# add the in-degree and out-degree of each node in h_net_sd to the seeds_nodes DataFrame
seeds_nodes['In-degree SD'] = seeds_nodes['ID'].map(h_net_sd_indegree)
seeds_nodes['Out-degree SD'] = seeds_nodes['ID'].map(h_net_sd_outdegree)

In [ ]:
h_nodes['In-degree'] = h_nodes['ID'].map(h_indegree)
h_nodes['Out-degree'] = h_nodes['ID'].map(h_outdegree)

In [ ]:
h_nodes['Cluster Coefficient'] = h_nodes['ID'].map(nx.clustering(G_h))

In [ ]:
# Convert the directed graph to an undirected graph
G_h_undirected = G_h.to_undirected()

# Compute the number of triangles for each node
h_nodes['Number of Triangles'] = h_nodes['ID'].map(nx.triangles(G_h_undirected))

In [ ]:
h_nodes

In [ ]:
nodes_by_cc_top10 = h_nodes[['ID','Organization Name','STATUS', 'Category', 'Cluster Coefficient']].sort_values(by='Cluster Coefficient', ascending=False).head(28)

nodes_by_cc_top10.to_latex('nodes_by_cc_top10.tex', index=False)

In [ ]:
# plot a doughnut chart of number of nodes with in-degree larger than 2, belonging to each category following color_map_cat

# count the number of nodes with in-degree larger than 2, belonging to each category
h_nodes_indegree = h_nodes[h_nodes['In-degree'] > 1]
h_nodes_indegree = h_nodes_indegree.groupby('Category').size()

# plot the doughnut chart
plt.figure(figsize=(10, 6))
plt.pie(h_nodes_indegree, labels=h_nodes_indegree.index, autopct='%1.1f%%', startangle=90, pctdistance=0.38, colors=[color_map_cat[cat] for cat in h_nodes_indegree.index])
centre_circle_in = plt.Circle((0, 0), 0.50, fc='white')
plt.gca().add_artist(centre_circle_in)
plt.title('Number of nodes with in-degree > 1, by category')
plt.show()


In [ ]:
# plot a doughnut chart of number of nodes with in-degree larger than 2, belonging to each category following color_map_cat
h_nodes['In-degree'] = h_nodes['ID'].map(h_indegree)
# count the number of nodes with in-degree larger than 2, belonging to each category
h_nodes_indegree = h_nodes[h_nodes['In-degree'] > 2]
h_nodes_indegree = h_nodes_indegree.groupby('Category').size()

# plot the doughnut chart
plt.figure(figsize=(10, 6))
plt.pie(h_nodes_indegree, labels=h_nodes_indegree.index, autopct='%1.1f%%', startangle=90, pctdistance=0.38, colors=[color_map_cat[cat] for cat in h_nodes_indegree.index])
centre_circle_in = plt.Circle((0, 0), 0.50, fc='white')
plt.gca().add_artist(centre_circle_in)
plt.title('Number of nodes with in-degree > 2, by category')
plt.show()


In [ ]:
# plot a doughnut chart of sum of in-degree of nodes belonging to each category following color_map_cat
h_nodes_indegree = h_nodes.groupby('Category')['In-degree'].sum()

# plot the doughnut chart
plt.figure(figsize=(10, 6))
plt.pie(h_nodes_indegree, labels=h_nodes_indegree.index, autopct='%1.1f%%', startangle=90, pctdistance=0.38, colors=[color_map_cat[cat] for cat in h_nodes_indegree.index])
centre_circle_in = plt.Circle((0, 0), 0.50, fc='white')
plt.gca().add_artist(centre_circle_in)
plt.title('Sum of in-degree of nodes, by category')
plt.show()


In [ ]:
nodes_by_indegree_top10 = h_nodes[['ID','Organization Name','STATUS', 'Category', 'In-degree']].sort_values(by='In-degree', ascending=False).head(10)

nodes_by_indegree_top10.to_latex('nodes_by_indegree_top10.tex', index=False)

In [ ]:
nodes_by_indegree = h_nodes[['ID','Organization Name','STATUS', 'Category', 'In-degree']].sort_values(by='In-degree', ascending=False)

In [ ]:
seeds_by_indegree_Tot_top10 = seeds_nodes[['ID','Label','Organization Name', 'Category', 'In-degree Tot']].sort_values('In-degree Tot', ascending=False).head(10)
seeds_by_outdegree_Tot_top10 = seeds_nodes[['ID','Label','Organization Name', 'Category', 'Out-degree Tot']].sort_values('Out-degree Tot', ascending=False).head(10)
seeds_by_outdegree_Seeds_top10 = seeds_nodes[['ID','Label','Organization Name', 'Category', 'Out-degree Seeds']].sort_values('Out-degree Seeds', ascending=False).head(10)
seeds_by_outdegree_SD_top10 = seeds_nodes[['ID','Label','Organization Name', 'Category', 'Out-degree SD']].sort_values('Out-degree SD', ascending=False).head(10)


# export the top 10 seeds by in-degree and out-degree as latex tables
seeds_by_indegree_Tot_top10.to_latex('seeds_by_indegree_Tot_top10.tex', index=False)
seeds_by_outdegree_Tot_top10.to_latex('seeds_by_outdegree_Tot_top10.tex', index=False)
seeds_by_outdegree_Seeds_top10.to_latex('seeds_by_outdegree_Seeds_top10.tex', index=False)
seeds_by_outdegree_SD_top10.to_latex('seeds_by_outdegree_SD_top10.tex', index=False)


# Twitter

In [ ]:
# Load the data
tw_nodes = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_nodes')
tw_edges = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_edges')
# Add Status
tw_nodes = tw_nodes.merge(h_nodes[['ID', 'STATUS']], left_on='ID', right_on='ID', how='left')

In [ ]:
# remove rows with 'Source'=='Target'
tw_edges = tw_edges[tw_edges['Source'] != tw_edges['Target']] 
tw_edges.reset_index(drop=True, inplace=True)

In [ ]:
# Create a directed graph for the Twitter data
G_tw = nx.DiGraph()

for index, row in tw_nodes.iterrows():
    G_tw.add_node(row['ID'], label=row['Label'], status=row['STATUS'], category=row['Category'], followers=row['Followers'])

# Add edges to the graph
for index, row in tw_edges.iterrows():
    G_tw.add_edge(row['Source'], row['Target'], label=row['Relationship'], date=row['Date'])

In [ ]:
# compute the degree of each node in G_tw
tw_indegree = dict(G_tw.in_degree())
tw_outdegree = dict(G_tw.out_degree())

In [ ]:
tw_nodes['In-degree'] = tw_nodes['ID'].map(tw_indegree)

In [ ]:
tw_nodes['Cluster Coefficient'] = tw_nodes['ID'].map(nx.clustering(G_tw))

# Convert the directed graph to an undirected graph
G_tw_undirected = G_tw.to_undirected()

# Compute the number of triangles for each node
tw_nodes['Number of Triangles'] = tw_nodes['ID'].map(nx.triangles(G_tw_undirected))

In [ ]:
tw_nodes

In [ ]:
nodes_by_cc_top18 = tw_nodes[['ID','Organization Name','STATUS', 'Category', 'Cluster Coefficient', 'Number of Triangles']].sort_values(by='Number of Triangles', ascending=False).head(18)
nodes_by_cc_top18['Cluster Coefficient'] = nodes_by_cc_top18['Cluster Coefficient'].round(3)
nodes_by_cc_top18.to_latex('nodes_by_cc_top18.tex', index=False)

In [ ]:
# plot a doughnut chart of number of nodes with in-degree larger than 2, belonging to each category following color_map_cat
tw_indegree = dict(G_tw.in_degree())
# count the number of nodes with in-degree larger than 2, belonging to each category
tw_nodes_indegree = tw_nodes[tw_nodes['In-degree'] > 1]
tw_nodes_indegree = tw_nodes_indegree.groupby('Category').size()

# plot the doughnut chart
plt.figure(figsize=(10, 6))
plt.pie(tw_nodes_indegree, labels=tw_nodes_indegree.index, autopct='%1.1f%%', startangle=90, pctdistance=0.38, colors=[color_map_cat[cat] for cat in tw_nodes_indegree.index])
centre_circle_in = plt.Circle((0, 0), 0.50, fc='white')
plt.gca().add_artist(centre_circle_in)
plt.title('Number of nodes with in-degree > 1, by category')
plt.show()

In [ ]:
# plot a doughnut chart of number of nodes with in-degree larger than 2, belonging to each category following color_map_cat
tw_indegree = dict(G_tw.in_degree())
# count the number of nodes with in-degree larger than 2, belonging to each category
tw_nodes_indegree = tw_nodes[tw_nodes['In-degree'] > 6]
tw_nodes_indegree = tw_nodes_indegree.groupby('Category').size()

# plot the doughnut chart
plt.figure(figsize=(10, 6))
plt.pie(tw_nodes_indegree, labels=tw_nodes_indegree.index, autopct='%1.1f%%', startangle=90, pctdistance=0.38, colors=[color_map_cat[cat] for cat in tw_nodes_indegree.index])
centre_circle_in = plt.Circle((0, 0), 0.50, fc='white')
plt.gca().add_artist(centre_circle_in)
plt.title('Number of nodes with in-degree > 6, by category')
plt.show()

In [ ]:
# plot a doughnut chart of the sum of in-degrees of nodes, belonging to each category following color_map_cat
tw_indegree = dict(G_tw.in_degree())
# sum the in-degrees of nodes, belonging to each category
tw_nodes_indegree = tw_nodes.groupby('Category').sum()['In-degree']

# plot the doughnut chart
plt.figure(figsize=(10, 6))
plt.pie(tw_nodes_indegree, labels=tw_nodes_indegree.index, autopct='%1.1f%%', startangle=90, pctdistance=0.38, colors=[color_map_cat[cat] for cat in tw_nodes_indegree.index])
centre_circle_in = plt.Circle((0, 0), 0.50, fc='white')
plt.gca().add_artist(centre_circle_in)
plt.title('Sum of in-degrees of nodes, by category')
plt.show()

In [ ]:
nodes_by_indegree_top10 = tw_nodes[['ID','Organization Name','STATUS', 'Category', 'In-degree']].sort_values(by='In-degree', ascending=False).head(10)

nodes_by_indegree_top10.to_latex('nodes_by_indegree_top10.tex', index=False)

In [ ]:
seeds_nodes = h_nodes[h_nodes['STATUS']=='IN']
#seeds_nodes['Stance']=[0,0,0,1,1,0,1,0,1,1,1,0,1,1,1,1]
seeds_nodes = seeds_nodes[['ID','Label', 'Organization Name', 'Category', 'Stance']]

In [ ]:
# create a directed graph among only nodes with 'STATUS' == 'IN'
tw_net_seeds = nx.DiGraph()
tw_net_sd = nx.DiGraph()
for i in range(tw_nodes.shape[0]):
    tw_net_sd.add_node(tw_nodes['ID'][i])
    if tw_nodes['STATUS'][i] == 'IN':
        tw_net_seeds.add_node(tw_nodes['ID'][i])
for i in range(tw_edges.shape[0]):
    if tw_edges['Source'][i] in tw_net_seeds.nodes() and tw_edges['Target'][i] in tw_net_seeds.nodes():
        tw_net_seeds.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])
    else:
        tw_net_sd.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])

# remove self loops
tw_net_seeds.remove_edges_from(nx.selfloop_edges(tw_net_seeds))
#tw_net_sd.remove_edges_from(nx.selfloop_edges(tw_net_sd))

In [ ]:
# create a directed graph among only nodes with 'STATUS' == 'IN'
tw_net_seeds = nx.DiGraph()
tw_net_sd = nx.DiGraph()
for i in range(tw_nodes.shape[0]):
    tw_net_sd.add_node(tw_nodes['ID'][i])
    if tw_nodes['STATUS'][i] == 'IN':
        tw_net_seeds.add_node(tw_nodes['ID'][i])
for i in range(tw_edges.shape[0]):
    if tw_edges['Source'][i] in tw_net_seeds.nodes() and tw_edges['Target'][i] in tw_net_seeds.nodes():
        tw_net_seeds.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])
    else:
        tw_net_sd.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])

# remove self loops
tw_net_seeds.remove_edges_from(nx.selfloop_edges(tw_net_seeds))
#tw_net_sd.remove_edges_from(nx.selfloop_edges(tw_net_sd))



In [ ]:
# compute the degree of each node in h_net_seeds
tw_net_seeds_indegree = dict(tw_net_seeds.in_degree())

# compute the out-degree of each node in h_net_seeds
tw_net_seeds_outdegree = dict(tw_net_seeds.out_degree())

# ----- In-degree Plot -----
# Sort the indegree dictionary by degree (largest to smallest)
sorted_indegree = sorted(tw_net_seeds_indegree.items(), key=lambda x: x[1], reverse=True)
# Unpack the sorted keys (node IDs) and their degrees
sorted_ids_indegree = [node for node, deg in sorted_indegree]
sorted_values_indegree = [deg for node, deg in sorted_indegree]

# Instead of organization names (which may be too long), use node IDs as labels
x_labels_indegree = sorted_ids_indegree
colors_indegree = [color_map_cat[h_nodes[h_nodes['ID'] == node]['Category'].values[0]] 
                   for node in sorted_ids_indegree]


# ----- Out-degree Plot -----
# Sort the outdegree dictionary by degree (largest to smallest)
sorted_outdegree = sorted(tw_net_seeds_outdegree.items(), key=lambda x: x[1], reverse=True)
# Unpack the sorted keys (node IDs) and their degrees
sorted_ids_outdegree = [node for node, deg in sorted_outdegree]
sorted_values_outdegree = [deg for node, deg in sorted_outdegree]

# Use node IDs as labels for shorter names
x_labels_outdegree = sorted_ids_outdegree
colors_outdegree = [color_map_cat[h_nodes[h_nodes['ID'] == node]['Category'].values[0]] 
                    for node in sorted_ids_outdegree]